# 🌿 Hierarchical Clustering
**ISLP Chapter 12 · Data Pattern Recognition for the Rest of Us**

> Hierarchical clustering builds a tree of clusters — you can explore any number of segments by cutting the dendrogram at different heights. Unlike K-means, you don't need to specify K upfront.

### Dataset
**S&P 500 Stock Returns — Sector Clustering**
We cluster major US stocks by their daily return correlations. Stocks that move together get clustered together — revealing sector structure and diversification opportunities. This is how portfolio managers and risk analysts actually use clustering.

### Time: ~55 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
import scipy.stats as stats
print("\u2713 Setup complete")

In [ ]:
# S&P 500 representative stocks — synthetic returns with realistic sector correlations
# Stocks within the same sector have correlated returns; cross-sector correlation is lower
np.random.seed(42)
n_days = 252  # one trading year

stocks = {
    'Tech':       ['AAPL','MSFT','GOOGL','META','NVDA','AMZN'],
    'Finance':    ['JPM','BAC','GS','MS','WFC','C'],
    'Healthcare': ['JNJ','PFE','UNH','ABT','MRK','TMO'],
    'Energy':     ['XOM','CVX','COP','SLB','PSX','VLO'],
    'Consumer':   ['PG','KO','PEP','WMT','COST','MCD'],
}

all_tickers = [t for tickers in stocks.values() for t in tickers]
sector_labels = {t: s for s, tickers in stocks.items() for t in tickers}

# Market factor (all stocks move somewhat together)
market_factor = np.random.normal(0, 0.008, n_days)

# Generate correlated returns
returns_data = {}
for sector, tickers in stocks.items():
    # Sector factor — stocks in same sector move together
    sector_factor = np.random.normal(0, 0.006, n_days)
    for ticker in tickers:
        # Individual noise
        idio = np.random.normal(0, 0.012, n_days)
        # Energy has negative correlation with market during crises
        mkt_beta = -0.3 if sector == 'Energy' else 0.8
        returns_data[ticker] = mkt_beta*market_factor + 0.6*sector_factor + idio

returns = pd.DataFrame(returns_data, index=pd.date_range('2023-01-01', periods=n_days, freq='B'))

print("S&P 500 Representative Portfolio")
print(f"  {len(all_tickers)} stocks across {len(stocks)} sectors")
print(f"  {n_days} trading days of simulated returns")
print(f"\nSectors: {list(stocks.keys())}")
print(f"\nAverage daily return: {returns.mean().mean()*100:.3f}%")
print(f"Average daily volatility: {returns.std().mean()*100:.2f}%")

## 📐 Part 1 — Clustering Stocks by Return Correlation

Instead of clustering raw return values, we cluster by **correlation distance**:
```
distance(i,j) = 1 - correlation(i,j)
```
Stocks that move together (high correlation) have low distance and cluster together.
This is the standard approach in portfolio risk analysis.

In [ ]:
# Compute correlation matrix and convert to distance
corr_matrix = returns.corr()
dist_matrix = 1 - corr_matrix

# Hierarchical clustering — complete linkage
Z = linkage(squareform(dist_matrix.values), method='complete')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(14, 5))
sector_colors = {'Tech':'#1e5fa8','Finance':'#e85d20','Healthcare':'#1a7a45',
                 'Energy':'#6b46c1','Consumer':'#0e7490'}
leaf_colors = [sector_colors[sector_labels[t]] for t in all_tickers]

dend = dendrogram(Z, labels=all_tickers, ax=ax, leaf_rotation=45,
                  color_threshold=0.7*max(Z[:,2]))
ax.set_title('Hierarchical Clustering of S&P 500 Stocks by Return Correlation\n'
             'Height = dissimilarity — stocks that merge early are most similar')
ax.set_ylabel('Correlation Distance (1 - correlation)')

# Add sector color legend
for sector, color in sector_colors.items():
    ax.plot([], [], 's', color=color, label=sector, markersize=8)
ax.legend(title='Actual Sector', loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()
print("\n\u2714 Stocks within the same sector cluster together at low height")
print("   Energy separates early (negative market correlation)")
print("   This confirms the sector structure without using any sector labels")

In [ ]:
# Cut dendrogram at different heights = different numbers of clusters
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, n_clusters in zip(axes, [3, 5, 7]):
    cluster_labels = fcluster(Z, n_clusters, criterion='maxclust')

    # Show correlation heatmap colored by cluster
    idx_sorted = np.argsort(cluster_labels)
    sorted_corr = corr_matrix.iloc[idx_sorted, idx_sorted]
    sorted_tickers = [all_tickers[i] for i in idx_sorted]

    im = ax.imshow(sorted_corr.values, cmap='RdYlGn', vmin=-0.5, vmax=1.0, aspect='auto')
    ax.set_xticks(range(len(sorted_tickers)))
    ax.set_yticks(range(len(sorted_tickers)))
    ax.set_xticklabels(sorted_tickers, rotation=90, fontsize=7)
    ax.set_yticklabels(sorted_tickers, fontsize=7)
    ax.set_title(f'{n_clusters} Clusters\n(sorted by cluster membership)')

    # Add cluster boundaries
    boundaries = np.where(np.diff(cluster_labels[idx_sorted]))[0] + 0.5
    for b in boundaries:
        ax.axhline(b, color='black', lw=1.5)
        ax.axvline(b, color='black', lw=1.5)

plt.colorbar(im, ax=axes[-1], label='Correlation', shrink=0.8)
plt.suptitle('Correlation Heatmaps at Different Cluster Cuts\n'
             'Green = high correlation, Red = negative correlation', fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Portfolio diversification insight
n_clusters = 5
cluster_labels = fcluster(Z, n_clusters, criterion='maxclust')
stock_clusters = pd.DataFrame({'Ticker': all_tickers,
                                'Sector': [sector_labels[t] for t in all_tickers],
                                'Cluster': cluster_labels})

print("=== Cluster Composition ===\n")
for cl in sorted(stock_clusters['Cluster'].unique()):
    members = stock_clusters[stock_clusters['Cluster']==cl]
    print(f"Cluster {cl} ({len(members)} stocks):")
    for _, row in members.iterrows():
        print(f"  {row['Ticker']:<8} [{row['Sector']}]")
    # Within-cluster avg correlation
    tickers = members['Ticker'].tolist()
    if len(tickers) > 1:
        avg_corr = corr_matrix.loc[tickers, tickers].values
        np.fill_diagonal(avg_corr, np.nan)
        print(f"  Avg within-cluster correlation: {np.nanmean(avg_corr):.3f}")
    print()

print("\u2714 Portfolio insight: pick ONE stock per cluster for maximum diversification")
print("   Stocks in the same cluster move together — owning multiple adds little diversification benefit")

In [ ]:
answers = {
    "q1": "",  # What is the key advantage of hierarchical over K-means clustering?
    "q2": "",  # What does complete linkage measure when merging two clusters?
    "q3": "",  # Why do we cluster by correlation rather than raw return levels?
    "q4": "",  # How does a dendrogram help choose the number of clusters?
    "q5": "",  # If two stocks are in the same cluster, what does that mean for portfolio diversification?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("\u274c Still empty: "+str(missing) if missing else "\u2705 Done! File \u2192 Save a copy in GitHub")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Hierarchical Clustering"
# @title 🤖 AI-Graded Submission — fill in the box and click ▶ Run
# @markdown ---
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2 (one-time):** Get a free AI grading key
# @markdown - Go to [aistudio.google.com](https://aistudio.google.com) — use your **personal Gmail** (not university email — many universities block AI Studio)
# @markdown - Click **Get API key → Create API key** → copy it
# @markdown - In Colab: click the **🔑 key icon** in the left sidebar → Add secret → Name: `GEMINI_API_KEY` → paste key → toggle ON
# @markdown - Done — this persists across all 30 notebooks automatically
# @markdown
# @markdown **Step 3:** Click ▶ Run on this cell for instant AI feedback

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ── DO NOT EDIT BELOW THIS LINE ──────────────────────────────────────────────
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Try to explain your reasoning in 1-2 sentences."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback on your answers."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with good detail. "
                             "Add a free Gemini key for AI analysis of your specific reasoning."),
                "tip": "Get a free key at aistudio.google.com using your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             f"Complete the remaining {n_total - n_answered} questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"\n  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    print(f"  Student  : @{username}" if username else
          "  Student  : \u26a0\ufe0f  Enter your GitHub username in the box above")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...\n")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)\n")
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 choose your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*